In [1]:
import utils
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split


**Reading the data**

In [2]:
train_df = pd.read_csv('data/train_data.csv')
train_df.head()

,title,link,domain,published_date,topic
0,"Tropical Storm Enteng out of PAR, but monsoon ...",https://rappler.com/nation/weather/tropical-st...,rappler.com,2020-08-09 09:41:24,NATION
1,Why iPhone 12 Will Be Another 'Defining Chapte...,https://www.theguardian.com/business/2020/aug/...,theguardian.com,2020-08-14 10:30:00,BUSINESS
2,Galway paddleboarders rescued after 15 hours i...,https://www.bbc.co.uk/news/world-europe-53767772,bbc.co.uk,2020-08-13 14:50:41,WORLD
3,Why iPhone 12 Will Be Another 'Defining Chapte...,https://www.globenewswire.com/news-release/202...,globenewswire.com,2020-08-11 08:28:00,BUSINESS
4,Our Favourite Android Games for 120Hz Phones,https://www.lifehacker.com.au/2020/08/our-favo...,lifehacker.com.au,2020-08-13 22:14:00,TECHNOLOGY


In [3]:
test_df = pd.read_csv('data/test_data.csv')
test_df.head()

,title,link,domain,published_date,topic
0,Dwarf planet Ceres is 'ocean world' with salty...,https://dunyanews.tv/en/WeirdNews/558898-Dwarf...,dunyanews.tv,2020-08-13 03:53:00,SCIENCE
1,'Very low double-digit' case growth needed to ...,https://www.afr.com/politics/federal/very-low-...,afr.com,2020-08-11 08:25:00,HEALTH
2,Daniel Wyllie to plead not guilty to domestic ...,https://www.news.com.au/entertainment/celebrit...,news.com.au,2020-08-12 00:52:50,ENTERTAINMENT
3,The Google Pixel 4a (5G) smartphone will be ba...,https://phonemantra.com/the-google-pixel-4a-5g...,phonemantra.com,2020-08-15 10:15:00,TECHNOLOGY
4,Why iPhone 12 Will Be Another 'Defining Chapte...,https://www.marketwatch.com/story/lear-swings-...,marketwatch.com,2020-08-04 10:14:00,BUSINESS


**Checking the ratio of training as well as test dataset**

**Percentage of training dataset**

In [4]:
train_df.topic.value_counts(normalize = True).sort_index().mul(100).round(1).astype(str) + '%' 

topic
BUSINESS         13.8%
ENTERTAINMENT    13.8%
HEALTH           13.8%
NATION           13.8%
SCIENCE           3.5%
SPORTS           13.8%
TECHNOLOGY       13.8%
WORLD            13.8%
Name: proportion, dtype: str

**Percentage of test dataset**

In [5]:
test_df.topic.value_counts(normalize = True).sort_index().mul(100).round(1).astype(str) + '%' 

topic
BUSINESS         13.8%
ENTERTAINMENT    13.8%
HEALTH           14.1%
NATION           13.5%
SCIENCE           3.6%
SPORTS           13.6%
TECHNOLOGY       13.7%
WORLD            13.8%
Name: proportion, dtype: str

**As we can see that, the data distribution of the topics among training and test dataset is unbalanced**
**So we must split the data in such that the training and test dataset has nearly balanced or balanced distribution of all the topics** 

In [6]:
total_df = pd.concat([train_df,test_df],ignore_index = True)  #merging the training as well as test dataset
total_df.shape

(108774, 5)

**Splitting of the dataset**

In [7]:
# Splitting the combined dataset to 60% train, 20% dev, and 20% test set inorder to produce a balanced split along the `topic` column.
train_df, test_df = train_test_split(total_df,test_size = 0.2)
train_df, dev_df = train_test_split(total_df,test_size = 0.2)

In [8]:
training_percentage = train_df.topic.value_counts(normalize = True).sort_index().mul(100).round(1).astype(str) + '%' 
print('percentage of topics in training dataset is:', training_percentage)
test_percentage = test_df.topic.value_counts(normalize = True).sort_index().mul(100).round(1).astype(str) + '%' 
print('percentage of topics in test dataset is:', test_percentage)

percentage of topics in training dataset is: topic
BUSINESS         13.7%
ENTERTAINMENT    13.8%
HEALTH           13.9%
NATION           13.8%
SCIENCE           3.5%
SPORTS           13.8%
TECHNOLOGY       13.8%
WORLD            13.8%
Name: proportion, dtype: str
percentage of topics in test dataset is: topic
BUSINESS         13.8%
ENTERTAINMENT    14.0%
HEALTH           14.4%
NATION           13.4%
SCIENCE           3.3%
SPORTS           14.0%
TECHNOLOGY       13.5%
WORLD            13.7%
Name: proportion, dtype: str


**Now the distribution of datas based on topic on both training as well as test dataset is almost balanced**

In [9]:
utils.save_data(train_df,'data' ,'train_data.csv')  #saving the new splitted training data
utils.save_data(test_df,'data' ,'test_data.csv')    #saving the new splitted test data

In [10]:
# Title length and vocabulary size used by the team for the prototype
MAX_LENGTH = 20
VOCAB_SIZE = 10000


**creating a lookup list for the topics as well as textvectorization for the titles**

In [11]:
title_preprocessor = tf.keras.layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LENGTH)
topic_lookup = tf.keras.layers.StringLookup(vocabulary='experiment/vocab/labels.txt', num_oov_indices=0)


In [ ]:
training_datas = train_df['title']  #extracting the title of the training datas
title_preprocessor.adapt(training_datas)  #generating the vocabulary from the new training data
utils.save_vocab(title_preprocessor,'vocabulary.txt')

**Conversion of the training ,cross_validation and test dataset into tensorflow dataset**

In [ ]:
train_dataset = utils.df_to_tfdata(train_df, topic_lookup, title_preprocessor, shuffle=True)
dev_dataset = utils.df_to_tfdata(dev_df, topic_lookup, title_preprocessor)
test_dataset = utils.df_to_tfdata(test_df, topic_lookup, title_preprocessor)

**Neural network model architecture for classification**

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=128),
    tf.keras.layers.Conv1D(128, 5, activation='relu'),
    tf.keras.layers.GlobalMaxPooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

